# **ADVANCE FEATURE ENGINEERING**

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
df = pd.read_csv('/content/preprocessed_data.csv')
df.head()

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,y,previous_contacted,previous_success,age_group
0,30,unemployed,married,primary,0,0.282211,0,0,cellular,19,oct,-0.888110,1,-1,0,0,0,0,Young
1,33,services,married,secondary,0,1.209846,1,1,cellular,11,may,0.166967,1,339,4,0,1,0,Adult
2,35,management,single,tertiary,0,0.130429,1,0,cellular,16,apr,-0.023029,1,330,1,0,1,0,Adult
3,30,management,married,tertiary,0,0.174936,1,1,unknown,3,jun,0.056370,4,-1,0,0,0,0,Young
4,59,blue-collar,married,secondary,0,-0.445382,1,0,unknown,5,may,0.196908,1,-1,0,0,0,0,Senior


In [4]:
df.shape

(4521, 19)

Customers contacted too often → lower response probability

Helps in risk-based prioritization

In [5]:
# Total lifetime contacts
df['total_contacts'] = df['campaign'] + df['previous']

# Contact pressure (current campaign intensity)
df['contact_pressure'] = df['campaign'] / (df['previous'] + 1)

# High contact flag (fatigue indicator)
df['high_contact_flag'] = (df['campaign'] > 3).astype(int)

Creates features that capture how recently a customer was contacted—flagging those never contacted, contacted recently (≤30 days), and grouping customers into recency categories (recent/warm/cold/never) to improve marketing response prediction.

In [6]:
# Never contacted before
df['never_contacted_before'] = (df['pdays'] == -1).astype(int)

# Recent contact (last 30 days)
df['recent_contact'] = df['pdays'].apply(lambda x: 1 if 0 <= x <= 30 else 0)

# Recency bucket
df['pdays_bucket'] = pd.cut(
    df['pdays'].replace(-1, 999),
    bins=[-1,30,90,180,999],
    labels=['recent','warm','cold','never']
)

Success in Previous Campaigns - Customers who responded earlier are **high-value targets**

In [7]:
# Previous success indicator
df['prev_success'] = (df['previous_success'] == 'success').astype(int)

# Any previous engagement
df['prev_contact_flag'] = (df['previous'] > 0).astype(int)

# Engagement strength
df['engagement_score'] = df['previous'] * df['prev_success']

In [8]:
df.head()

,age,job,marital,education,default,balance,housing,loan,contact,day,...,age_group,total_contacts,contact_pressure,high_contact_flag,never_contacted_before,recent_contact,pdays_bucket,prev_success,prev_contact_flag,engagement_score
0,30,unemployed,married,primary,0,0.282211,0,0,cellular,19,...,Young,1,1.0,0,1,0,never,0,0,0
1,33,services,married,secondary,0,1.209846,1,1,cellular,11,...,Adult,5,0.2,0,0,0,never,0,1,0
2,35,management,single,tertiary,0,0.130429,1,0,cellular,16,...,Adult,2,0.5,0,0,0,never,0,1,0
3,30,management,married,tertiary,0,0.174936,1,1,unknown,3,...,Young,4,4.0,1,1,0,never,0,0,0
4,59,blue-collar,married,secondary,0,-0.445382,1,0,unknown,5,...,Senior,1,1.0,0,1,0,never,0,0,0


In [9]:
df.shape

(4521, 28)

In [10]:
df.to_csv("final_data.csv", index=False)